# 05 — Segmentation Training

A **transparent, compact training loop** for one scenario × architecture, using the
same building blocks as `stages/training/train_segmentation.py` (channel selection,
spatial block split, per-band normalization, model, loss, eval) with an explicit
epoch loop so each step is visible.

- **scenarios** (`SCENARIO`): `single_date`, `mt_ndvi`, `gsi`, `rf`
- **architectures** (`ARCH`): `deeplabv3plus_cbam` (ResNet-50), `segformer` (MiT-B2)
- **normalization** (`NORM`): `percentile` *(main)*, `minmax`, `zscore`
- **loss** (`LOSS`): `dynamic_balanced` = **DECB-CE** *(main)*, `focal_tversky`, `wce`
- gsi/rf channels come from `select_{sel}_direct_s0.5.json` (notebook 04)

> This demo omits the production extras in `main()` (preload cache, augmentation,
> class-balanced sampler, MLflow logging, checkpointing, per-patch viz, NDVI
> analysis). For the full 4×2 matrix / ablations / seed-grid, use `T.main()` or the
> CLI — see the appendix. **GPU strongly recommended.**

In [ ]:
# Register this repo as `crop_mapping_pipeline` regardless of checkout dir name,
# and silence MLflow telemetry.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo   :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# reused building blocks (identical to train_segmentation.run_experiment)
from crop_mapping_pipeline.stages.training.train_segmentation import (
    build_model, compute_class_weights, evaluate_test_set,
    NormalizedDataset, _filter_s2_by_band_indices,
)
from crop_mapping_pipeline.stages.training.experiments.base import build_local_band_map
from crop_mapping_pipeline.stages.training.experiments.single_date import build_single_date_indices
from crop_mapping_pipeline.stages.training.experiments.mt_ndvi import build_naive_multitemporal_indices
from crop_mapping_pipeline.stages.training.experiments.feature_selection import build_direct_indices
from crop_mapping_pipeline.stages.data.spatial_split import _block_spatial_split
from crop_mapping_pipeline.stages.training.normalization import load_or_compute_norm_stats
from crop_mapping_pipeline.stages.training.losses import (
    build_wce, build_dynamic_balanced, build_focal_tversky,
)
from crop_mapping_pipeline.stages.selection.band_scoring import get_train_year_inputs
from geoai.geoai.train import RasterPatchDataset

# ── knobs ──
SCENARIO = 'gsi'               # single_date | mt_ndvi | gsi | rf
ARCH     = 'segformer'         # deeplabv3plus_cbam | segformer
NORM     = 'percentile'        # percentile (main) | minmax | zscore
LOSS     = 'dynamic_balanced'  # dynamic_balanced (DECB-CE, main) | focal_tversky | wce
THRESH   = 0.5                 # gsi/rf normalized-score threshold
EPOCHS   = 3                   # smoke test; full runs use C.MAX_EPOCHS

DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
_yr, s2_paths, cdl_path = get_train_year_inputs()   # valid-filtered S2 + CDL (v6.1 = 2024)
print(f'{len(s2_paths)} S2 dates | device {DEVICE} | {SCENARIO}/{ARCH}/{NORM}/{LOSS}')

## 1. Channel selection for the scenario

`single_date`/`mt_ndvi` = NDVI-picked dates × all bands; `gsi`/`rf` = the threshold selection JSON from notebook 04. Returns global channel indices into the date×band stack.

In [ ]:
names, band_to_idx, date_to_idx, mmdd_to_date = build_local_band_map(s2_paths)

if SCENARIO == 'single_date':
    idx, ch_names, _ = build_single_date_indices(
        date_to_idx, band_to_idx, s2_paths=s2_paths, cdl_path=cdl_path)
elif SCENARIO == 'mt_ndvi':
    idx, ch_names, _ = build_naive_multitemporal_indices(
        date_to_idx, band_to_idx, s2_paths=s2_paths, cdl_path=cdl_path)
else:
    sel   = 'gsi_direct' if SCENARIO == 'gsi' else 'rf_direct'
    jpath = C.PROCESSED_DIR / f'select_{sel}_s{THRESH:g}.json'   # from notebook 04
    idx, ch_names = build_direct_indices(jpath, mmdd_to_date, band_to_idx,
                                         selector_name=SCENARIO, subset_k=None)
print(f'{SCENARIO}: {len(idx)} channels')

## 2. Dataset + spatial block split + normalization

Patchify (`RasterPatchDataset`), compute per-band norm stats, wrap with `NormalizedDataset`, then split into train/val/test by **whole spatial blocks** (no patch-adjacency leakage).

In [ ]:
s2_filt, local_idx = _filter_s2_by_band_indices(s2_paths, idx)
ds_raw = RasterPatchDataset(
    s2_paths=s2_filt, cdl_path=cdl_path,
    patch_size=C.PATCH_SIZE, stride=C.STRIDE,
    keep_classes=C.KEEP_CLASSES, remap_lut=C.REMAP_LUT,
    min_valid_frac=C.MIN_VALID_FRAC, band_indices=local_idx,
)
band_pct = load_or_compute_norm_stats(NORM, s2_filt, Path(s2_filt[0]).parent)
ds = NormalizedDataset(ds_raw, band_percentiles=band_pct, norm_mode=NORM)

tr, va, te, info = _block_spatial_split(
    [ds_raw], C.BLOCK_SIZE, C.VAL_FRAC, C.TEST_FRAC,
    C.NUM_CLASSES, C.SEED, min_class_frac=C.MIN_CLASS_FRAC,
)
train_dl = DataLoader(Subset(ds, tr), batch_size=C.BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_dl   = DataLoader(Subset(ds, va), batch_size=C.BATCH_SIZE, shuffle=False, num_workers=2)
test_dl  = DataLoader(Subset(ds, te), batch_size=C.BATCH_SIZE, shuffle=False, num_workers=2) if te else None
print(f'{len(tr)} train / {len(va)} val / {len(te)} test patches ({len(local_idx)} channels)')

## 3. Model + loss + optimizer

`build_model` (same encoders as production); loss per `LOSS`; optimizer from `ARCH_CFG`.

In [ ]:
model = build_model(ARCH, len(local_idx), C.NUM_CLASSES).to(DEVICE)

cw, counts = compute_class_weights(return_counts=True)
if LOSS == 'wce':
    criterion = build_wce(cw.to(DEVICE))
elif LOSS == 'focal_tversky':
    criterion = build_focal_tversky(class_counts=counts)
else:  # dynamic_balanced (DECB-CE)
    criterion = build_dynamic_balanced(num_classes=C.NUM_CLASSES)
if hasattr(criterion, 'to'):
    criterion = criterion.to(DEVICE)

cfg = C.ARCH_CFG[ARCH]
if cfg['optimizer'] == 'sgd':
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg['lr'],
                                momentum=0.9, weight_decay=cfg['weight_decay'])
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'])
print(f'{ARCH}: {sum(p.numel() for p in model.parameters()):,} params | loss={LOSS} | opt={cfg["optimizer"]}')

## 4. Training loop

Explicit epoch loop: forward → loss → backward → step, then validation mIoU/mF1 via `evaluate_test_set`. (Production adds a class-balanced sampler + augmentation + LR schedule + early stopping — omitted here for clarity.)

In [ ]:
best_miou = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for imgs, masks in train_dl:
        imgs = torch.nan_to_num(imgs).to(DEVICE)
        masks = masks.to(DEVICE).long()
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward(); optimizer.step()
        running += loss.item()
    val = evaluate_test_set(model, val_dl, C.NUM_CLASSES, DEVICE)
    best_miou = max(best_miou, val['miou'])
    print(f'epoch {epoch:>2}: train_loss {running/len(train_dl):.4f} | '
          f'val mIoU {val["miou"]:.4f}  mF1 {val["mf1"]:.4f}  OA {val["oa"]:.4f}')
print(f'best val mIoU: {best_miou:.4f}')

## 5. Test evaluation (held-out blocks)

In [ ]:
if test_dl is not None:
    res = evaluate_test_set(model, test_dl, C.NUM_CLASSES, DEVICE)
    print(f'TEST  mIoU {res["miou"]:.4f} | mF1 {res["mf1"]:.4f} | OA {res["oa"]:.4f}')
    for c, iou in zip(C.KEEP_CLASSES, res['per_class_iou'][1:]):   # [0] = background
        print(f'  {C.CDL_CLASS_NAMES[c]:14s} IoU {iou:.3f}')
else:
    print('No test split (TEST_FRAC=0).')

## Appendix — full matrix / ablations / seed-grid

The compact loop above trains one config. For the production runs (with MLflow, checkpoints, augmentation, sampler, LR schedule, early stopping) use `main()` or the CLI:

In [ ]:
from crop_mapping_pipeline.stages.training import train_segmentation as T

# Full 4×2 matrix, main norm + loss:
# T.main(exps=['single_date','mt_ndvi','gsi','rf'],
#        archs=['deeplabv3plus_cbam','segformer'],
#        score_threshold=0.5, norm_mode='percentile', loss='dynamic_balanced')

# Normalization ablation:
# for nm in ('percentile','minmax','zscore'):
#     T.main(exps=['gsi'], archs=['segformer'], norm_mode=nm, loss='dynamic_balanced')

# Loss ablation:
# for ls in ('dynamic_balanced','focal_tversky','wce'):
#     T.main(exps=['gsi'], archs=['segformer'], norm_mode='percentile', loss=ls)

# Seed-grid (CLI):
#   python stages/training/train_segmentation.py --exp gsi --arch segformer \
#          --score-threshold 0.5 --seed-grid 42 123 456 789